# R1 & Janus

In [1]:
import csv
import itertools
import base64
import json
import subprocess
import ollama
import torch
from transformers import AutoModelForCausalLM
from janus.models import MultiModalityCausalLM, VLChatProcessor
from janus.utils.io import load_pil_images

print("Starting script execution...")

# -----------------------------------
# Configuration
# -----------------------------------
RUN_SINGLE_COMBINATION = False  # Set to True for testing with one combination
OLLAMA_MODEL_NAME = "deepseek-r1:7b"  # Model name for context analysis
JANUS_MODEL_PATH = "deepseek-ai/Janus-Pro-1B"  # Model path for image analysis

print(f"Configuration loaded - Running in {'single' if RUN_SINGLE_COMBINATION else 'full'} combination mode")
print(f"Using Ollama model: {OLLAMA_MODEL_NAME}")
print(f"Using Janus model path: {JANUS_MODEL_PATH}")

# -----------------------------------
# Janus Model Setup
# -----------------------------------
def setup_janus_model():
    print("Setting up Janus model...")
    print("Loading VL chat processor...")
    vl_chat_processor = VLChatProcessor.from_pretrained(JANUS_MODEL_PATH)
    tokenizer = vl_chat_processor.tokenizer
    print("Loading VL GPT model...")
    vl_gpt = AutoModelForCausalLM.from_pretrained(
        JANUS_MODEL_PATH, trust_remote_code=True
    )
    vl_gpt = vl_gpt.to(torch.bfloat16).cpu().eval()
    print("Janus model setup complete")
    return vl_chat_processor, tokenizer, vl_gpt

# -----------------------------------
# Janus Image Analysis
# -----------------------------------
def analyze_image_with_janus(image_path, trend_prompt, vl_chat_processor, tokenizer, vl_gpt):
    print(f"\nAnalyzing image: {image_path}")
    conversation = [
        {
            "role": "<|User|>",
            "content": f"<image_placeholder>\n{trend_prompt}",
            "images": [image_path],
        },
        {"role": "<|Assistant|>", "content": ""},
    ]

    print("Loading PIL images...")
    pil_images = load_pil_images(conversation)
    print("Preparing inputs...")
    prepare_inputs = vl_chat_processor(
        conversations=conversation, images=pil_images, force_batchify=True
    ).to("cpu")

    print("Preparing input embeddings...")
    inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)

    print("Generating analysis...")
    outputs = vl_gpt.language_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=prepare_inputs.attention_mask,
        pad_token_id=tokenizer.eos_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=512,
        do_sample=False,
        use_cache=True,
    )

    analysis = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
    print("Analysis generation complete")
    return analysis

# -----------------------------------
# Utility Functions
# -----------------------------------
def get_llm_response(prompt):
    """
    Retrieve text-only response from DeepSeek model for context analysis.
    """
    print("\nGenerating LLM response...")
    try:
        print("Sending prompt to Ollama...")
        response = ollama.generate(
            model=OLLAMA_MODEL_NAME,
            prompt=prompt
        )
        print("Successfully received response from Ollama")
        return response['response']
    except Exception as e:
        print(f"Error generating LLM response: {e}")
        return None

# New function: Load existing rows from CSV into a dictionary
def load_existing_rows(file_name, headers):
    existing_data = {}
    try:
        with open(file_name, mode='r', encoding='utf-8') as f:
            reader = csv.reader(f)
            # Read the header row
            file_header = next(reader, None)
            # If the headers don't match (e.g. different columns), skip loading
            if file_header == headers:
                for row in reader:
                    if not row:
                        continue
                    # Combination Description is row[0]
                    combination = row[0]
                    existing_data[combination] = row
    except FileNotFoundError:
        print("No existing CSV found; will create a new one.")
    return existing_data

# Updated function: Either update existing row or add a new one, then rewrite CSV
def append_analysis_to_csv(
    file_name,
    combination_desc,
    context_prompt,
    context_response,
    trend_analysis_prompts,
    trend_analysis_responses,
    existing_data,
    headers
):
   

    # If this combination already exists, update it
    if combination_desc in existing_data:
        row = existing_data[combination_desc]
    else:
        # Create a new row filled with empty strings
        row = [""] * len(headers)
        row[0] = combination_desc

    # Fill in context prompt & response if empty
    if not row[1]:
        row[1] = context_prompt
    if not row[2]:
        row[2] = context_response

   
    for i in range(1, 11):
        prompt_col = 3 + (i - 1) * 2
        analysis_col = 4 + (i - 1) * 2
        # If it's empty, fill it
        if not row[prompt_col] and not row[analysis_col]:
            # We assume that we created the lists in the same order
            # so index i-1 in the lists belongs to image i
            row[prompt_col] = trend_analysis_prompts[i - 1]
            row[analysis_col] = trend_analysis_responses[i - 1]

    # Save it back in existing_data
    existing_data[combination_desc] = row

    # Rewrite entire CSV with updated data
    with open(file_name, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        for cmb_desc, cmb_row in existing_data.items():
            writer.writerow(cmb_row)
    print("CSV updated successfully")

# -----------------------------------
# Load Files & Define Templates
# -----------------------------------
print("\nLoading input files...")
try:
    with open('TXT/narrativeCombined200.txt', 'r', encoding='utf-8') as file:
        parameters = file.read()
        print("Successfully loaded parameters file")
except Exception as e:
    print(f"Error loading parameters file: {e}")
    parameters = ""

try:
    with open('TXT/finalDocumentation200.txt', 'r', encoding='utf-8') as file:
        documentation = file.read()
        print("Successfully loaded documentation file")
except Exception as e:
    print(f"Error loading documentation file: {e}")
    documentation = ""

context_prompt_template = f"""
Your goal is to explain an agent-based model. Your explanation must include the context of the model, its goals, and key parameters.
For each parameter, include the range of values (if provided) and the value that we have set. Do not write any summary or conclusion.
The parameters of the model are as follows:
"{parameters}"

The context and goals of the model are as follows:
"{documentation}"

Your task is NOT to understand how to set up experiments. Do not give me steps. Your task is only to explain the model. You must summarize the text that I gave you by including the context, goals, and parameters.
"""

print("Context prompt template created")

# Plot descriptions
plot_descriptions = [
    "This plot represents the average initial risk attitude of all social learning agents, used to store and report their initial disposition to deviate from the recommended number of livestock.",
    "This plot represents the average degree of deviation from the recommended number of livestock for all extension agents during the simulation.",
    "This plot represents the average amount of reserve biomass available across all patches, indicating the overall biomass condition in the model's grazing system.",
    "This plot represents the average amount of actual available green biomass across all patches, accounting for reductions due to grazing.",
    "This plot represents the average total number of livestock, including both healthy livestock and those that have been destocked, across all households.",
    "This plot represents the average number of livestock that have been destocked, or removed due to insufficient resources, across all households.",
    "This plot represents the inequality in the distribution of healthy livestock across all households.",
    "This plot represents the number of households that have engaged in social learning, indicated by the variable household-SL being true.",
    "This plot represents the average cumulative number of livestock that have been destocked across all households.",
    "This plot represents the number of households that follow the E-RO strategy, indicating households with non learning agents."
]


print(f"Loaded {len(plot_descriptions)} plot descriptions")

trend_analysis_prompt_template = """
We have a plot from repeated simulations of an agent-based model. Your goal is to describe the trends in detail from the plot, mentioning key time steps and values taken by the metric, and interpreting them based on the model's context.
The report must objectively describe the trends in the data without addressing the quality of the simulation.
Do not refer to the plot or any visual in your description. If a plot has very simple dynamics, simply state them without expanding.
"""

# Define features
features = {
    "role": "You are an expert in grazing systems with a statistic background.",
    "example": """Here is an example of the style of the report: “ The total number of earthquakes recorded in the region starts at 5,000 then it declines rapidly over the first 400 time steps. This initial decline could be attributed to the depletion of immediate stress points within the fault lines, as the most susceptible areas release their pent-up energy early in the simulation. It increases briefly at around 500 steps, likely due to the aftershocks and secondary stress points being triggered as the system seeks a new equilibrium. The simulation ends with the number of earthquakes near zero, indicating that the majority of the stress has been released and the system has stabilized.

The total seismic activity follows the same pattern, starting at 10,000 events and declining steadily. This steady decline reflects a systematic reduction in seismic energy over time, as the energy distribution within the tectonic plates becomes more uniform. There is low variance across simulation runs in the first 100 steps, but deviations become noticeable afterward. This suggests that while the initial reactions to stress are highly predictable, as time progresses, the system's complexity introduces more variability. This variability could be due to differences in secondary stress points and the non-linear nature of seismic energy release over time.“""",
    "insights": "When summarizing trends, provide brief insights about their implications for decision makers."
}

print("Features dictionary loaded")

# -----------------------------------
# Main Function
# -----------------------------------
def main():
    print("\n=== Starting Main Execution ===")
    file_name = 'CSV/Refining/R1JanusAllResponses.csv'

    # Define the headers we expect
    headers = ['Combination Description', 'Context Prompt', 'Context Response']
    for i in range(1, 11):
        headers.extend([f'Plot {i} Prompt', f'Plot {i} Analysis'])

    # Load existing data or create file if not exists
    existing_data = load_existing_rows(file_name, headers)
    if not existing_data:
        # If file does not exist, create it with headers
        try:
            with open(file_name, mode='x', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow(headers)
            print("Created new CSV file with headers")
        except FileExistsError:
            pass
        # Load again to ensure we are consistent
        existing_data = load_existing_rows(file_name, headers)

    # Setup Janus model
    print("\nInitializing Janus model...")
    vl_chat_processor, tokenizer, vl_gpt = setup_janus_model()

    # Generate feature combinations
    print("\nGenerating feature combinations...")
    feature_keys = list(features.keys())
    all_combinations = []
    for i in range(len(feature_keys) + 1):
        all_combinations.extend(itertools.combinations(feature_keys, i))
    print(f"Generated {len(all_combinations)} feature combinations")

    for idx, combination in enumerate(all_combinations, 1):
        combination_desc = "+".join(combination) if combination else "None"
        print(f"\n=== Processing Combination {idx}/{len(all_combinations)} ===")
        print(f"Current combination: {combination_desc}")

        # Check if we already have a row for this combination in existing_data
        row_data = existing_data.get(combination_desc, None)

        # Determine if everything is done for this combination
        # If row_data exists, check if all columns are filled
        if row_data:
            # If all columns up to the last prompt/analysis are non-empty, skip entirely
            all_filled = True
            for col in row_data[1:]:  # ignoring combination_desc
                if not col:
                    all_filled = False
                    break
            if all_filled:
                print(f"Combination '{combination_desc}' is already fully processed. Skipping.")
                continue

        print("Building prompts...")

        # Build context prompt and trend analysis base prompt
        context_prompt_parts = [context_prompt_template]
        trend_analysis_prompt_parts = [trend_analysis_prompt_template]

        if "role" in combination:
            context_prompt_parts.insert(0, features["role"])
            trend_analysis_prompt_parts.append(features["role"])
        if "example" in combination:
            trend_analysis_prompt_parts.append(features["example"])
        if "insights" in combination:
            trend_analysis_prompt_parts.append(features["insights"])

        context_prompt = "\n\n".join(context_prompt_parts)
        trend_analysis_prompt_base = "\n\n".join(trend_analysis_prompt_parts)

        # If context response or prompt is missing, generate it
        if not row_data or (row_data and not row_data[1] and not row_data[2]):
            print("Getting context analysis from DeepSeek...")
            context_response = get_llm_response(context_prompt)
            print("Context analysis complete")
        else:
            # If row_data exists, fill from it
            existing_context_prompt = row_data[1]
            existing_context_response = row_data[2]
            context_prompt = existing_context_prompt if existing_context_prompt else context_prompt
            context_response = existing_context_response if existing_context_response else get_llm_response(context_prompt)

        trend_analysis_prompts = []
        trend_analysis_responses = []

        # Process images with Janus if needed
        if RUN_SINGLE_COMBINATION:
            print("\nRunning single combination test...")
            # Only do Plot 1 if not already done
            if not row_data or (row_data and (not row_data[3] and not row_data[4])):
                image_path = "Images/1.png"
                trend_prompt = trend_analysis_prompt_base + "\n\n" + plot_descriptions[0]
                analysis = analyze_image_with_janus(
                    image_path,
                    trend_prompt,
                    vl_chat_processor,
                    tokenizer,
                    vl_gpt
                )
                trend_analysis_prompts.append(trend_prompt)
                trend_analysis_responses.append(analysis)
            else:
                # If already done, just reuse that row’s data
                trend_analysis_prompts.append(row_data[3])
                trend_analysis_responses.append(row_data[4])

            print("Single combination test complete")
        else:
            print("\nProcessing all images...")
            for i in range(1, 11):
                prompt_col = 3 + (i - 1) * 2
                analysis_col = 4 + (i - 1) * 2
                already_done = False
                existing_prompt = ""
                existing_analysis = ""
                if row_data:
                    existing_prompt = row_data[prompt_col]
                    existing_analysis = row_data[analysis_col]
                    if existing_prompt and existing_analysis:
                        already_done = True

                if already_done:
                    print(f"Plot {i} already has data. Skipping Janus run.")
                    trend_analysis_prompts.append(existing_prompt)
                    trend_analysis_responses.append(existing_analysis)
                else:
                    print(f"\nProcessing image {i}/10 for combination '{combination_desc}'")
                    image_path = f"Images/{i}.png"
                    trend_prompt = trend_analysis_prompt_base + "\n\n" + plot_descriptions[i - 1]
                    analysis = analyze_image_with_janus(
                        image_path,
                        trend_prompt,
                        vl_chat_processor,
                        tokenizer,
                        vl_gpt
                    )
                    trend_analysis_prompts.append(trend_prompt)
                    trend_analysis_responses.append(analysis)
                    print(f"Image {i} processing complete")

        # Now append/update the CSV
        append_analysis_to_csv(
            file_name,
            combination_desc,
            context_prompt,
            context_response,
            trend_analysis_prompts,
            trend_analysis_responses,
            existing_data,
            headers
        )

        if RUN_SINGLE_COMBINATION:
            print("\nSingle combination mode - ending execution")
            break

    print("\n=== Main Execution Complete ===")

if __name__ == "__main__":
    print("\n=== Script Execution Starting ===")
    main()
    print("\n=== Script Execution Complete ===")


/Applications/anaconda3/envs/janusV3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python version is above 3.10, patching the collections module.


/Applications/anaconda3/envs/janusV3/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:590: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


Starting script execution...
Configuration loaded - Running in full combination mode
Using Ollama model: deepseek-r1:7b
Using Janus model path: deepseek-ai/Janus-Pro-1B

Loading input files...
Successfully loaded parameters file
Successfully loaded documentation file
Context prompt template created
Loaded 10 plot descriptions
Features dictionary loaded

=== Script Execution Starting ===

=== Starting Main Execution ===
No existing CSV found; will create a new one.
Created new CSV file with headers

Initializing Janus model...
Setting up Janus model...
Loading VL chat processor...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
Some kwargs in processor config are unused and will not have any effect: ignore_id, sft_format, image_tag, add_speci

Loading VL GPT model...
Janus model setup complete

Generating feature combinations...
Generated 8 feature combinations

=== Processing Combination 1/8 ===
Current combination: None
Building prompts...
Getting context analysis from DeepSeek...

Generating LLM response...
Sending prompt to Ollama...
Successfully received response from Ollama
Context analysis complete

Processing all images...

Processing image 1/10 for combination 'None'

Analyzing image: Images/1.png
Loading PIL images...
Preparing inputs...
Preparing input embeddings...
Generating analysis...
Analysis generation complete
Image 1 processing complete

Processing image 2/10 for combination 'None'

Analyzing image: Images/2.png
Loading PIL images...
Preparing inputs...
Preparing input embeddings...
Generating analysis...
Analysis generation complete
Image 2 processing complete

Processing image 3/10 for combination 'None'

Analyzing image: Images/3.png
Loading PIL images...
Preparing inputs...
Preparing input embeddings..